### DASHBOARD DE ACOMPANHAMENTO DE INVESTIMENTOS

Objetivo: automatizar a visualização de investimentos a partir de índices atualizados.

---

Ideias principais:

Criar um dashboard em PowerBI para visualização dos investimentos
- Porporção dos tipos de investimentos;
- Variação da cotação de cada investimento;
- DY;
- Histórico de variação dos preços;
- Renda fixa/Renda variável.

Back-end com API's Python.

---

Questões:
Como inserir novos dados? Tabela de compra/venda como input para o Python --> usa csv antigo e o novo, e cria um novo.

---

Adicional: implementar resumo dos investimentos com LLM.

---
### Podemos usar um SQL (SQLite ou PostgreSQL) para armazenar os dados de input e output. Talve
Input:
- ticker da ação;
- quantidade de cotas;
- valor comprado (ou quantidade de cotas);
- taxas de compra/venda;
- taxas de adm;

In [4]:
%pip install python-dotenv requests

  Using cached python_dotenv-1.2.3-py3-none-any.whl.metadata (29 kB)
Using cached python_dotenv-1.2.3-py3-none-any.whl (22 kB)
Note: you may need to restart the kernel to use updated packages.


In [40]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import requests
import warnings

from dotenv import load_dotenv
from pathlib import Path

In [63]:
# Reprodutibilidade
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Caminhos do projeto
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = DATA_DIR / "historico_operacoes.csv"
OUTPUT_PATH = DATA_DIR / "carteira_atualizada.csv"

# Configurações do Pandas
pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

# Configurações dos gráficos
plt.style.use("default")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True

# Ignorar warnings desnecessários
warnings.filterwarnings("ignore")

In [66]:
load_dotenv()

BASE_URL = "https://brapi.dev/api/v2"
TOKEN = os.getenv("BRAPI_TOKEN")

def get_stock_quote(symbols):
    url = f"{BASE_URL}/stocks/quote"

    params = {
        "symbols": ",".join(symbols),
        "token": TOKEN
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    # Pega o JSON da resposta
    json_response = response.json()
    resultados = json_response.get("results", [])

    # EXTRAÇÃO DO DICIONÁRIO 'data':
    # Percorre cada ação encontrada e extrai apenas a parte que importa (a chave 'data')
    dados_limpos = [acao['data'] for acao in resultados if 'data' in acao]
    
    return dados_limpos

# Chamando a função
dados_acoes = get_stock_quote(["PETR4", "VALE3", "ITUB4"])

# Cria o DataFrame passando a lista de dicionários extraídos
df = pd.DataFrame(dados_acoes)

# visualização de algumas colunas
df_resumo = df[['shortName', 'regularMarketPrice', 'regularMarketDayHigh', 'regularMarketDayLow','regularMarketChangePercent','logourl']]
df_resumo

,shortName,regularMarketPrice,regularMarketDayHigh,regularMarketDayLow,regularMarketChangePercent,logourl
0,PETR4,42.47,42.76,41.72,0.90,https://icons.brapi.dev/icons/PETR4.svg
1,VALE3,71.41,71.80,70.69,0.15,https://icons.brapi.dev/icons/VALE3.svg
2,ITUB4,38.38,38.98,38.17,-1.59,https://icons.brapi.dev/icons/ITUB4.svg


### Principais dados a serem exportados da API
- Symbol (shortname) que representa o ticker da ação, por exemplo: ITUB4
- regularMarketPrice que mostra o preço atual da ação, permitindo calcular o valor atualizado da carteira
- regularMarketChangePercent indicando a variação percentual do dia. Podemos ainda adicionar o preço máximo e mínimo da ação no dia (high and low)
- logourl isso será importante no PowerBI, mostra o URL da logo da empresa.

Um dos pontos que a API Brapi não fornece é o dividendo anual (DY - dividend yield)

In [ ]:
try:
    df_operacoes = pd.read_csv(INPUT_PATH)
    print("Arquivo de operações carregado com sucesso!")
except FileNotFoundError:
    print(f"⚠️ Arquivo {INPUT_PATH.name} não encontrado. Usando dados simulados temporários.")
    # Como não achou o arquivo, criamos um DataFrame vazio só para não dar erro
    df_operacoes = pd.DataFrame() 

# Simulando o dataframe de posições já agrupadas (quantidade atual e preço médio):
dados_posicao = {
    'Ticker': ['PETR4', 'VALE3', 'ITUB4'],
    'Qtd_Cotas': [200, 50, 100],
    'Preco_Medio': [38.50, 68.00, 32.50]
}
df_carteira = pd.DataFrame(dados_posicao)

# ==========================================
# 3. CONSULTA À API E CRUZAMENTO
# ==========================================
# dados_api = get_stock_quote(df_carteira['Ticker'].tolist()) # Sua função aqui
# (Simulando a resposta da API para o código rodar independente)
dados_api = [
    {'shortName': 'PETR4', 'regularMarketPrice': 42.47, 'regularMarketChangePercent': 0.9},
    {'shortName': 'VALE3', 'regularMarketPrice': 71.41, 'regularMarketChangePercent': 0.15},
    {'shortName': 'ITUB4', 'regularMarketPrice': 38.38, 'regularMarketChangePercent': -1.59}
]

df_mercado = pd.DataFrame(dados_api)[['shortName', 'regularMarketPrice', 'regularMarketChangePercent']]

df_final = pd.merge(df_carteira, df_mercado, left_on='Ticker', right_on='shortName', how='left')

# ==========================================
# 4. MÉTRICAS E EXPORTAÇÃO
# ==========================================
df_final['Valor_Investido'] = df_final['Qtd_Cotas'] * df_final['Preco_Medio']
df_final['Valor_Atual'] = df_final['Qtd_Cotas'] * df_final['regularMarketPrice']
df_final['Lucro_Prejuizo_R$'] = df_final['Valor_Atual'] - df_final['Valor_Investido']
df_final['Rentabilidade_%'] = (df_final['Lucro_Prejuizo_R$'] / df_final['Valor_Investido']) * 100

# Limpando colunas duplicadas
df_final = df_final.drop(columns=['shortName'])

# Exportando usando o OUTPUT_PATH (Caminho + Nome do arquivo já definidos lá em cima)
df_final.to_csv(OUTPUT_PATH, index=False)

print(f"✅ Dados atualizados com sucesso! Salvos em: {OUTPUT_PATH}")

Arquivo de operações carregado com sucesso!
✅ Dados atualizados com sucesso! Salvos em: e:\linkedin\Projetos\investment_monitoring\data\carteira_atualizada.csv
